In [ ]:
import pandas as pd
import plotly.express as px

space_df = pd.read_csv('data/Data_Streamlit/Space_Limpa_Com_Risco_Individual.csv')
sites_df = pd.read_csv('data/launch_sites.csv')

mapeamento_manual_definitivo = {
    # --- Os Grandes Polos ---
    'Florida': (28.56, -80.58), 'California': (34.63, -120.61), 
    'French Guiana': (5.24, -52.77), 'Jiuquan Satellite Launch Center': (40.96, 100.28),
    'Xichang Satellite Launch Center': (28.24, 102.02), 'Taiyuan Satellite Launch Center': (38.84, 111.60),
    'Kiritimati Launch Area': (1.88, -157.40), 'Virginia': (37.83, -75.48), 
    'Texas': (25.99, -97.16), 'M? hia Peninsula': (-39.26, 177.86), 
    'Yasny Cosmodrome': (51.22, 59.83), 'San Marco Launch Platform': (-2.94, 40.21),
    'Marshall Islands': (8.72, 167.73), 'Wenchang Satellite Launch Center': (19.61, 110.95),
    'RAAF Woomera Range Complex': (-30.95, 136.53),
    
    # --- Os 37 Locais Específicos e Históricos ---
    'Vostochny Cosmodrome': (51.88, 128.33),
    'Svobodny Cosmodrome': (51.88, 128.33),
    'Spaceport America': (32.99, -106.97),
    'Algeria': (30.89, -3.03), # Base de Hammaguir
    'Sohae Satellite Launching Station': (39.66, 124.70),
    'Alaska': (57.43, -152.33), # Pacific Spaceport Complex
    'Barents Sea Launch Area': (69.50, 34.00), # Lançamento de Submarino
    'Maranh?œo': (-2.31, -44.36), # Alcântara (Corrigindo o erro de digitação da base)
    'Tonghae Satellite Launching Ground': (40.85, 129.66),
    'Base Aerea de Gando': (27.93, -15.38),
    'Kauai': (22.02, -159.78),
    'Launch Plateform': (34.90, 121.19), # Mar Amarelo
    'Tai Rui Barge': (34.90, 121.19)
}

# Faz a junção inicial
df_mapa = space_df.merge(
    sites_df[['site_name', 'latitude', 'longitude']], 
    left_on='Estado_Regiao', 
    right_on='site_name', 
    how='left'
)

# Preenche os buracos usando o dicionário definitivo
for idx, row in df_mapa.iterrows():
    if pd.isna(row['latitude']) and row['Estado_Regiao'] in mapeamento_manual_definitivo:
        lat, lon = mapeamento_manual_definitivo[row['Estado_Regiao']]
        df_mapa.at[idx, 'latitude'] = lat
        df_mapa.at[idx, 'longitude'] = lon

# Preencher Custos vazios com o valor mediano apenas para o gráfico não dar erro no tamanho das bolhas
custo_mediano = df_mapa['Custo_da_Missao'].median()
df_mapa['Custo_Para_Grafico'] = df_mapa['Custo_da_Missao'].fillna(custo_mediano)


fig_mapa = px.scatter_mapbox(
    df_mapa, 
    lat="latitude", 
    lon="longitude", 
    
    # Informações que aparecem ao passar o rato (mouse):
    hover_name="Carga_Util", 
    hover_data={
        "Nome_da_Empresa": True,
        "Estado_Regiao": True,
        "Modelo_Foguete": True,
        "Ano": True,
        "Status_da_Missao": True,
        "Probabilidade_Sucesso": ":.2f", # Formata a probabilidade para 2 casas decimais
        "latitude": False, 
        "longitude": False,
        "Custo_Para_Grafico": False # Esconde esta coluna auxiliar
    },
    
    # Configurações de Cor e Tamanho:
    color="Probabilidade_Sucesso",          # A cor indica o Risco/Sucesso
    color_continuous_scale=px.colors.sequential.Plasma, # Escala visual de calor
    size="Custo_Para_Grafico",              # O tamanho da bolha indica o custo
    size_max=15,                            # Tamanho máximo da bolha
    
    # Estilo base do mapa:
    zoom=1.2, 
    height=750,
    title="<b>Mapeamento Espacial Global: Análise de Risco e Custo (100% dos Dados)</b>"
)

# Estilo de mapa claro (necessita de ligação à internet)
fig_mapa.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":50,"l":0,"b":0}
)

# Mostrar o mapa!
fig_mapa.show()

# Print de verificação
print(f"Total de lançamentos mapeados com sucesso: {len(df_mapa)}")
print(f"Lançamentos sem coordenadas: {df_mapa['latitude'].isna().sum()}")

C:\Users\jzirn\AppData\Local\Temp\ipykernel_10824\808584699.py:65: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_mapa = px.scatter_mapbox(


Total de lançamentos mapeados com sucesso: 4324
Lançamentos sem coordenadas: 13


In [14]:
df_mapa

,Nome_da_Empresa,Status_do_Foguete,Custo_da_Missao,Status_da_Missao,Pais,Estado_Regiao,Ano,Mes,Dia,Hora,...,Alvo_Sucesso,Empresa_Cod,Pais_Cod,Probabilidade_Sucesso,Probabilidade_Falha,Previsao_IA_Status,site_name,latitude,longitude,Custo_Para_Grafico
0,SpaceX,StatusActive,50.00,Success,USA,Florida,2020,8.0,7.0,5.0,...,1,46.0,16.0,0.942317,0.057683,1,NaN,28.56,-80.58,50.00
1,CASC,StatusActive,29.75,Success,China,Jiuquan Satellite Launch Center,2020,8.0,6.0,4.0,...,1,7.0,2.0,0.792976,0.207024,1,NaN,40.96,100.28,29.75
2,SpaceX,StatusActive,56.50,Success,USA,Texas,2020,8.0,4.0,23.0,...,1,46.0,16.0,0.955834,0.044166,1,NaN,25.99,-97.16,56.50
3,Roscosmos,StatusActive,65.00,Success,Kazakhstan,Baikonur Cosmodrome,2020,7.0,30.0,21.0,...,1,42.0,9.0,0.929818,0.070182,1,Baikonur Cosmodrome,45.92,63.34,65.00
4,ULA,StatusActive,145.00,Success,USA,Florida,2020,7.0,30.0,11.0,...,1,48.0,16.0,0.935731,0.064269,1,NaN,28.56,-80.58,145.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4319,US Navy,StatusRetired,150.00,Failure,USA,Florida,1958,2.0,5.0,7.0,...,0,50.0,16.0,0.034684,0.965316,0,NaN,28.56,-80.58,150.00
4320,AMBA,StatusRetired,150.00,Success,USA,Florida,1958,2.0,1.0,3.0,...,1,1.0,16.0,0.532172,0.467828,1,NaN,28.56,-80.58,150.00
4321,US Navy,StatusRetired,150.00,Failure,USA,Florida,1957,12.0,6.0,16.0,...,0,50.0,16.0,0.470281,0.529719,0,NaN,28.56,-80.58,150.00
4322,RVSN USSR,StatusRetired,150.00,Success,Kazakhstan,Baikonur Cosmodrome,1957,11.0,3.0,2.0,...,1,40.0,9.0,0.657642,0.342358,1,Baikonur Cosmodrome,45.92,63.34,150.00


In [15]:
df_mapa = df_mapa.drop(columns=['site_name', 'Custo_Para_Grafico'])
# Salvando o DataFrame completo (com 100% das latitudes e longitudes) num novo arquivo CSV
df_mapa.to_csv('Space_Limpa_Com_Risco_Individual.csv', index=False)

print("Arquivo 'Space_Limpa_Com_Risco_Individual.csv' salvo com sucesso!")

Arquivo 'Space_Limpa_Com_Risco_Individual.csv' salvo com sucesso!
